In [ ]:
import ast, hashlib
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from tqdm import tqdm
import networkx as nx
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
N_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3  # 이 값 초과 등장 이미지는 "우연 재사용"으로 간주, 그룹핑에서 제외

DATA_DIR = '...'  # 실제 경로로 교체
train = pd.read_csv(f'{DATA_DIR}/train.csv')
train['answer_tuple'] = train['Answer'].apply(lambda x: str(ast.literal_eval(x)))


# --- Step 1: 이미지 해시 계산 ---
def compute_md5(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def get_all_image_records(df, img_dir):
    records = []
    for _, row in df.iterrows():
        for i in range(1, 5):
            path = f"{img_dir}/{row['Id']}/{row[f'Input_{i}']}"
            records.append({'sample_id': row['Id'], 'path': path})
    return records

train_records = get_all_image_records(train, f'{DATA_DIR}/train')
paths = [r['path'] for r in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(tqdm(executor.map(compute_md5, paths), total=len(paths)))
for r, h in zip(train_records, hashes):
    r['hash'] = h


# --- Step 2: 3개 초과 반복 등장 이미지는 제외하고 그래프로 클러스터링 ---
hash_counter = defaultdict(set)
for r in train_records:
    hash_counter[r['hash']].add(r['sample_id'])

hash_to_samples = {h: s for h, s in hash_counter.items() if len(s) <= MAX_SAMPLES_PER_HASH}

G = nx.Graph()
G.add_nodes_from(train['Id'].unique())
for h, sample_ids in hash_to_samples.items():
    sample_ids = list(sample_ids)
    for i in range(len(sample_ids)):
        for j in range(i + 1, len(sample_ids)):
            G.add_edge(sample_ids[i], sample_ids[j])

components = list(nx.connected_components(G))
id_to_cluster = {sid: cid for cid, comp in enumerate(components) for sid in comp}
train['cluster_id'] = train['Id'].map(id_to_cluster)

print(f"총 {len(components)}개 클러스터 (대부분 크기 1, 일부만 2~14개)")


# --- Step 3: 그룹 단위로 Stratified K-Fold 분할 ---
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train, train['answer_tuple'], groups=train['cluster_id'])):
    train.loc[val_idx, 'fold'] = fold


# --- Step 4: 검증 (아래 두 값이 반드시 이렇게 나와야 정상) ---
cluster_fold_counts = train.groupby('cluster_id')['fold'].nunique()
assert (cluster_fold_counts > 1).sum() == 0, "같은 클러스터가 여러 fold에 걸쳐있음 — leakage 위험!"
print(train['fold'].value_counts().sort_index())  # 각 fold 약 1900개씩 균등해야 함